# Fleet encoder: spatial layout vs. embedding space

Loads a trained `FleetEncoder` checkpoint, picks one turn from a replay, and shows the same set of fleets in two views:

1. **2D board** — actual `(x, y)` positions, sun + planets for context, velocity arrows, colored by mission type.
2. **3D embedding** — the per-fleet `d_model=64` projection reduced to 3 dims via PCA, same color scheme.

If pretraining worked, fleets that share a mission type should cluster together in embedding space even when their board positions don't.

## Setup

In [ ]:
import sys
from pathlib import Path
REPO = Path.cwd()
while not (REPO / 'agents').is_dir() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
print('repo:', REPO)

import gzip, json, math
import numpy as np
import torch
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers 3d projection)

from agents.transformer_v1.encoder.fleet_encoder import FleetEncoder
from agents.transformer_v1.featurizer import (
    FleetTracker, featurize_fleets, FLEET_RAW_DIM,
)
from agents.transformer_v1.featurizer.fleet_featurizer import (
    _MISSION_ATTACK, _MISSION_REINFORCE, _MISSION_CAPTURE_NEUTRAL,
    _MISSION_COMET_CHASE, _MISSION_TRANSIT_LOST,
)

## Load the trained encoder

The pretrain wrapper saved `{encoder.*, heads.<name>.*}` into the same state dict; we strip the `encoder.` prefix and discard the heads.

In [ ]:
CKPT = REPO / 'data' / 'encoder_runs' / '20260429-194952' / 'fleet_encoder_best.pt'
ckpt = torch.load(CKPT, map_location='cpu', weights_only=False)
d_model = ckpt['config']['d_model']

encoder = FleetEncoder(d_model=d_model)
encoder_sd = {
    k.removeprefix('encoder.'): v
    for k, v in ckpt['model'].items()
    if k.startswith('encoder.')
}
missing = encoder.load_state_dict(encoder_sd, strict=True)
encoder.eval()
print(f'loaded encoder (d_model={d_model}, epoch={ckpt["epoch"]}); load_state_dict: {missing}')

## Pick a replay turn

Pick any `.json.gz` under `data/replays/` and a step. Mid-game (around the median step) usually has the most interesting fleet composition.

In [ ]:
REPLAY = REPO / 'data' / 'replays' / 'local' / 'phy_rnd_2p' / 'local00010001_2_0.json.gz'
STEP = None  # None → auto-pick the step with the most fleets

with gzip.open(REPLAY, 'rt') as fh:
    replay = json.load(fh)
steps = replay['steps']

if STEP is None:
    counts = [len((s[0]['observation'].get('fleets') or [])) if s else 0 for s in steps]
    STEP = int(np.argmax(counts))
    print(f'auto-picked STEP={STEP} (max fleets at this turn: {counts[STEP]})')

obs = steps[STEP][0]['observation']
n_players = len(steps[0])
print(f'replay: {REPLAY.name}  step={STEP}/{len(steps)}  '
      f'players={n_players}  fleets={len(obs.get("fleets") or [])}  '
      f'planets={len(obs.get("planets") or [])}')

## Featurize and encode

Replay the tracker from step 0 up to `STEP` so `age_since_launch` is correct, then featurize and run through the encoder.

In [ ]:
tracker = FleetTracker()
for s in steps[:STEP + 1]:
    if s:
        tracker.observe(s[0]['observation'])

features, mask, records = featurize_fleets(
    obs,
    learner_slot=0,
    tracker=tracker,
    max_fleets=256,
    num_players=n_players,
)
n_fleets = int(mask.sum())
real_features = features[:n_fleets]              # (F, FLEET_RAW_DIM)
with torch.no_grad():
    embeds = encoder(real_features.unsqueeze(0)).squeeze(0)  # (F, d_model)
embeds_np = embeds.numpy()
print(f'features {tuple(real_features.shape)} → embeddings {tuple(embeds_np.shape)}')

## 2D board view

Sun in yellow at (50, 50). Planets are circles colored by owner (-1 = neutral grey), sized by `√ships`. Fleets are markers at their `(x, y)`, colored by mission type, with a thin arrow showing velocity.

In [ ]:
MISSION_NAMES = {
    _MISSION_ATTACK: 'attack',
    _MISSION_REINFORCE: 'reinforce',
    _MISSION_CAPTURE_NEUTRAL: 'capture_neutral',
    _MISSION_COMET_CHASE: 'comet_chase',
    _MISSION_TRANSIT_LOST: 'transit_lost',
}
MISSION_COLORS = {
    _MISSION_ATTACK: '#d62728',          # red
    _MISSION_REINFORCE: '#2ca02c',       # green
    _MISSION_CAPTURE_NEUTRAL: '#1f77b4', # blue
    _MISSION_COMET_CHASE: '#9467bd',     # purple
    _MISSION_TRANSIT_LOST: '#7f7f7f',    # grey
}
OWNER_COLORS = {0: '#1f77b4', 1: '#d62728', 2: '#2ca02c', 3: '#9467bd', -1: '#cccccc'}

fleet_colors = [MISSION_COLORS[r.mission_type_coarse] for r in records[:n_fleets]]

fig, ax = plt.subplots(figsize=(8, 8))
ax.set_xlim(0, 100); ax.set_ylim(0, 100); ax.set_aspect('equal')
ax.set_facecolor('#0b1020')
ax.add_patch(plt.Circle((50, 50), 10, color='#ffd700', alpha=0.85, label='sun'))

for p in obs['planets']:
    pid, owner, x, y, radius, ships, _prod = p[:7]
    ax.add_patch(plt.Circle((x, y), max(0.8, math.sqrt(max(1, ships)) * 0.4),
                            color=OWNER_COLORS.get(owner, '#cccccc'),
                            alpha=0.55, ec='white', lw=0.6))
    ax.text(x, y, str(pid), color='white', fontsize=7, ha='center', va='center')

for i, rec in enumerate(records[:n_fleets]):
    ax.scatter([rec.x], [rec.y], c=fleet_colors[i], s=40, ec='white', lw=0.5, zorder=3)
    ax.arrow(rec.x, rec.y, rec.vx * 1.5, rec.vy * 1.5,
             head_width=1.0, head_length=1.2,
             color=fleet_colors[i], alpha=0.8, length_includes_head=True, zorder=2)
    ax.text(rec.x + 1.2, rec.y + 1.2, str(i), color='white', fontsize=6, alpha=0.7)

# legend
from matplotlib.lines import Line2D
handles = [Line2D([0], [0], marker='o', color='w', markerfacecolor=c, markersize=8, label=name)
           for name, c in zip(MISSION_NAMES.values(), MISSION_COLORS.values())]
ax.legend(handles=handles, loc='upper right', fontsize=8, facecolor='#11183a', labelcolor='white')
ax.set_title(f'{REPLAY.name}  step {STEP}  ({n_fleets} fleets)', color='white')
ax.tick_params(colors='white')
for s in ax.spines.values(): s.set_color('white')
plt.tight_layout(); plt.show()

## 3D embedding view (PCA)

PCA the `(F, 64)` embeddings down to 3 components. Same color = same mission type as the board view, so you can tell at a glance whether the encoder pulled mission-mates together regardless of where they sit on the board.

In [ ]:
# PCA without sklearn — center, SVD, take top 3 components.
Z = embeds_np - embeds_np.mean(axis=0, keepdims=True)
_, S, Vt = np.linalg.svd(Z, full_matrices=False)
comps = Z @ Vt[:3].T   # (F, 3)
explained = (S[:3] ** 2) / (S ** 2).sum()
print('explained variance ratio (top-3):',
      ', '.join(f'{v:.2%}' for v in explained),
      f'  total={explained.sum():.2%}')

fig = plt.figure(figsize=(10, 8))
ax3 = fig.add_subplot(111, projection='3d')
ax3.set_facecolor('#0b1020'); fig.patch.set_facecolor('#0b1020')
for mt, name in MISSION_NAMES.items():
    idx = [i for i, r in enumerate(records[:n_fleets]) if r.mission_type_coarse == mt]
    if not idx:
        continue
    ax3.scatter(comps[idx, 0], comps[idx, 1], comps[idx, 2],
                c=MISSION_COLORS[mt], s=45, ec='white', lw=0.4,
                label=f'{name} (n={len(idx)})')
for i in range(n_fleets):
    ax3.text(comps[i, 0], comps[i, 1], comps[i, 2], str(i),
             color='white', fontsize=6, alpha=0.7)
ax3.set_xlabel(f'PC1 ({explained[0]:.1%})', color='white')
ax3.set_ylabel(f'PC2 ({explained[1]:.1%})', color='white')
ax3.set_zlabel(f'PC3 ({explained[2]:.1%})', color='white')
ax3.tick_params(colors='white')
ax3.set_title(f'fleet embeddings (PCA→3d), step {STEP}', color='white')
ax3.legend(facecolor='#11183a', labelcolor='white', loc='upper left', fontsize=8)
plt.tight_layout(); plt.show()

## Optional — recolor by another label

Same 3-d projection, recolored. Useful for sanity-checking that the encoder also separates fleets along axes other than the one above.

In [ ]:
# Color by target_relationship: 0=none, 1=neutral, 2=friendly, 3=hostile.
from agents.transformer_v1.featurizer.fleet_featurizer import _target_relationship
REL_NAMES = {0: 'none', 1: 'neutral', 2: 'friendly', 3: 'hostile'}
REL_COLORS = {0: '#7f7f7f', 1: '#1f77b4', 2: '#2ca02c', 3: '#d62728'}

rels = [_target_relationship(r) for r in records[:n_fleets]]

fig = plt.figure(figsize=(10, 8))
ax3 = fig.add_subplot(111, projection='3d')
ax3.set_facecolor('#0b1020'); fig.patch.set_facecolor('#0b1020')
for rv, name in REL_NAMES.items():
    idx = [i for i, r in enumerate(rels) if r == rv]
    if not idx:
        continue
    ax3.scatter(comps[idx, 0], comps[idx, 1], comps[idx, 2],
                c=REL_COLORS[rv], s=45, ec='white', lw=0.4,
                label=f'{name} (n={len(idx)})')
ax3.set_xlabel('PC1', color='white'); ax3.set_ylabel('PC2', color='white'); ax3.set_zlabel('PC3', color='white')
ax3.tick_params(colors='white')
ax3.set_title('same PCA, recolored by target_relationship', color='white')
ax3.legend(facecolor='#11183a', labelcolor='white', loc='upper left', fontsize=8)
plt.tight_layout(); plt.show()

---
# Why PCA didn't separate the classes

PCA is variance-greedy, not class-greedy. The encoder has 13 supervised heads — 5 of them continuous regression (`recon_x/y/vx/vy`, `recon_ships_log`) — so the dominant variance directions are mostly position/kinematics. Mission-separating axes exist (the head hit 100% test acc), they're just not in the top 3 variance directions.

Plus: **within a single turn**, fleets share spatial context. Many were launched from the same planet on similar trajectories, so spatial variance dwarfs categorical variance.

Two things help:

1. **Aggregate across many turns / replays** — diversifies the sample so categorical variance has room to show.
2. **Switch to a supervised projection (LDA)** — finds directions that maximize between-class / within-class variance ratio. Guaranteed to surface separation if it exists in 64-d.

## Aggregate embeddings across many steps and replays

In [ ]:
import random
REPLAY_GLOB = list((REPO / 'data' / 'replays').rglob('*.json.gz'))
rng = random.Random(0)
rng.shuffle(REPLAY_GLOB)
REPLAY_GLOB = REPLAY_GLOB[:8]   # 8 replays is plenty for this viz
STEPS_PER_REPLAY = 6

all_embeds, all_missions, all_rels = [], [], []
for path in REPLAY_GLOB:
    with gzip.open(path, 'rt') as fh:
        rep = json.load(fh)
    rsteps = rep['steps']
    npl = len(rsteps[0])
    candidates = list(range(5, len(rsteps) - 1))
    chosen = sorted(rng.sample(candidates, min(STEPS_PER_REPLAY, len(candidates))))
    tr = FleetTracker()
    last_seen = -1
    for t in chosen:
        # Bring tracker forward so age_since_launch is correct.
        for u in range(last_seen + 1, t + 1):
            if rsteps[u]:
                tr.observe(rsteps[u][0]['observation'])
        last_seen = t
        ob = rsteps[t][0]['observation']
        if not (ob.get('fleets') or []):
            continue
        f, m, recs = featurize_fleets(
            ob, learner_slot=0, tracker=tr, max_fleets=512, num_players=npl,
        )
        nf = int(m.sum())
        if nf == 0:
            continue
        with torch.no_grad():
            e = encoder(f[:nf].unsqueeze(0)).squeeze(0).numpy()
        all_embeds.append(e)
        all_missions.extend(r.mission_type_coarse for r in recs[:nf])
        all_rels.extend(_target_relationship(r) for r in recs[:nf])

E = np.concatenate(all_embeds, axis=0)
missions = np.asarray(all_missions)
rels = np.asarray(all_rels)
from collections import Counter
print(f'pooled {len(REPLAY_GLOB)} replays \u00d7 {STEPS_PER_REPLAY} steps -> {E.shape[0]} embeddings (d={E.shape[1]})')
print('  mission counts:', Counter(missions.tolist()))
print('  relation counts:', Counter(rels.tolist()))

## LDA → 3D, supervised on mission type

Multi-class LDA projects to up to `C-1` axes (4 for mission, 3 for relation). It explicitly maximizes between-class spread, so the 3D scatter should show clean clusters if the encoder really separates mission types in the 64-d space.

In [ ]:
def lda_project(X, y, k=3):
    """Multi-class LDA without sklearn. Returns top-k components."""
    classes = np.unique(y)
    overall_mean = X.mean(axis=0)
    Sw = np.zeros((X.shape[1], X.shape[1]))
    Sb = np.zeros((X.shape[1], X.shape[1]))
    for c in classes:
        Xc = X[y == c]
        nc = Xc.shape[0]
        if nc < 2:
            continue
        mc = Xc.mean(axis=0)
        Sw += (Xc - mc).T @ (Xc - mc)
        diff = (mc - overall_mean).reshape(-1, 1)
        Sb += nc * (diff @ diff.T)
    Sw_reg = Sw + 1e-3 * np.eye(Sw.shape[0])
    M = np.linalg.solve(Sw_reg, Sb)
    eigvals, eigvecs = np.linalg.eig(M)
    order = np.argsort(-eigvals.real)[:k]
    W = eigvecs[:, order].real
    return X @ W, eigvals.real[order]

lda_xyz, lda_eig = lda_project(E, missions, k=3)
print('LDA top-3 eigenvalues:', np.round(lda_eig, 3))

fig = plt.figure(figsize=(10, 8))
ax3 = fig.add_subplot(111, projection='3d')
ax3.set_facecolor('#0b1020'); fig.patch.set_facecolor('#0b1020')
for mt, name in MISSION_NAMES.items():
    sel = missions == mt
    if not sel.any():
        continue
    ax3.scatter(lda_xyz[sel, 0], lda_xyz[sel, 1], lda_xyz[sel, 2],
                c=MISSION_COLORS[mt], s=15, ec='white', lw=0.2, alpha=0.85,
                label=f'{name} (n={int(sel.sum())})')
ax3.set_xlabel('LDA1', color='white'); ax3.set_ylabel('LDA2', color='white'); ax3.set_zlabel('LDA3', color='white')
ax3.tick_params(colors='white')
ax3.set_title(f'fleet embeddings (LDA \u2192 3d, supervised on mission), n={E.shape[0]}', color='white')
ax3.legend(facecolor='#11183a', labelcolor='white', loc='upper left', fontsize=8)
plt.tight_layout(); plt.show()

## t-SNE → 3D (label-agnostic, nonlinear)

t-SNE preserves local neighborhoods; clusters that are linearly tangled in 64-d often pop apart here. Coloring by mission afterwards is just for reading the picture — t-SNE never sees the labels.

Falls back to PCA on the pooled set if `sklearn` isn't installed.

In [ ]:
try:
    from sklearn.manifold import TSNE
    tsne = TSNE(n_components=3, perplexity=30, init='pca', random_state=0)
    xyz = tsne.fit_transform(E)
    method = 't-SNE'
except ImportError:
    Z = E - E.mean(axis=0, keepdims=True)
    _, _, Vt = np.linalg.svd(Z, full_matrices=False)
    xyz = Z @ Vt[:3].T
    method = 'PCA (sklearn unavailable)'

fig = plt.figure(figsize=(10, 8))
ax3 = fig.add_subplot(111, projection='3d')
ax3.set_facecolor('#0b1020'); fig.patch.set_facecolor('#0b1020')
for mt, name in MISSION_NAMES.items():
    sel = missions == mt
    if not sel.any():
        continue
    ax3.scatter(xyz[sel, 0], xyz[sel, 1], xyz[sel, 2],
                c=MISSION_COLORS[mt], s=15, ec='white', lw=0.2, alpha=0.85,
                label=f'{name} (n={int(sel.sum())})')
ax3.set_xlabel(f'{method}-1', color='white'); ax3.set_ylabel(f'{method}-2', color='white'); ax3.set_zlabel(f'{method}-3', color='white')
ax3.tick_params(colors='white')
ax3.set_title(f'fleet embeddings ({method} \u2192 3d), n={E.shape[0]}', color='white')
ax3.legend(facecolor='#11183a', labelcolor='white', loc='upper left', fontsize=8)
plt.tight_layout(); plt.show()